# Project: Linear Algebra Applications with NumPy

Explore real-world linear algebra: solving systems of equations, PCA from scratch, image compression with SVD, and PageRank simulation.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.set_printoptions(precision=4, suppress=True)
print('NumPy version:', np.__version__)

## 1. Solving Systems of Linear Equations

In [ ]:
A = np.array([[10, -5, 0],
              [-5, 15, -10],
              [0, -10, 20]])
b = np.array([10, 0, 0])
currents = np.linalg.solve(A, b)
print('Loop currents (A):')
for i, c in enumerate(currents, 1):
    print(f'  I{i} = {c:.4f}')
print(f'Residual: {A @ currents - b}')

## 2. Principal Component Analysis from Scratch

In [ ]:
np.random.seed(42)
n = 300
X = np.random.multivariate_normal(
    mean=[50, 30, 20],
    cov=[[100, 80, 40], [80, 100, 50], [40, 50, 60]],
    size=n)

X_centered = X - X.mean(axis=0)
cov = (X_centered.T @ X_centered) / (n - 1)
eigenvalues, eigenvectors = np.linalg.eigh(cov)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

X_pca = X_centered @ eigenvectors[:, :2]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X[:, 0], X[:, 1], alpha=0.6, c='steelblue')
axes[0].set_title('Original Data')
axes[0].grid(True, alpha=0.3)
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, c='coral')
axes[1].set_title('PCA Projection (2 components)')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Explained variance: {eigenvalues[:2].sum()/eigenvalues.sum():.3f}')

## 3. Image Compression with SVD

In [ ]:
from sklearn.datasets import load_sample_image
china = load_sample_image('china.jpg')
img = np.mean(china, axis=2) / 255.0
U, S, Vt = np.linalg.svd(img, full_matrices=False)

ranks = [5, 20, 50, 100]
fig, axes = plt.subplots(1, len(ranks)+1, figsize=(20, 4))
axes[0].imshow(img, cmap='gray')
axes[0].set_title(f'Original (rank={img.shape[0]})')
axes[0].axis('off')
for i, r in enumerate(ranks):
    recon = U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]
    axes[i+1].imshow(recon, cmap='gray')
    storage = r*(img.shape[0]+img.shape[1]+1)/(img.shape[0]*img.shape[1])*100
    axes[i+1].set_title(f'Rank={r} ({storage:.1f}% storage)')
axes[i+1].axis('off')
plt.tight_layout()
plt.show()

## 4. PageRank via Power Iteration

In [ ]:
A = np.array([[0,1,1,0,0],[1,0,0,1,0],[0,1,0,1,1],[0,0,1,0,1],[1,0,0,1,0]], dtype=float)
d = 0.85
n_pages = 5
M = np.zeros_like(A)
for i in range(n_pages):
    out = A[i].sum()
    if out > 0:
        M[:, i] = A[i] / out
r = np.ones(n_pages) / n_pages
for _ in range(50):
    r_new = d * M @ r + (1-d)/n_pages
    if np.linalg.norm(r_new - r) < 1e-6:
        break
    r = r_new
print('PageRank scores:')
for i, score in enumerate(r):
    print(f'  Page {i+1}: {score:.4f}')
print(f'Top page: Page {np.argmax(r)+1}')

## Summary

- Solving linear systems with np.linalg.solve
- PCA via eigen decomposition
- Image compression with SVD
- PageRank via power iteration